# Detección de Reclamos de Seguridad en Reviews de Billeteras Móviles

**Objetivo:** Analizar los comentarios descargador y guardados en tabla y agregar la columna `ReclamoSeguridad` que indica si el review menciona un problema de seguridad (suplantación de identidad, fraude, hackeo, robo de cuenta, etc.).

**Stack:**
- **LangGraph** → orquesta el flujo de clasificación como un grafo de estados.
- **Spark + Pandas** → carga de la tabla origen y construcción del DataFrame final.

**Flujo lógico:**

`Tabla Delta → Pandas → LangGraph (LLM por cada comentario) → Columna ReclamoSeguridad → Spark DataFrame`

## 1. Instalación de dependencias
* Instalacion de LangGraph.
* Instalacion del SDK de Databricks para llamar al modelo fundacional y habilitar el wrapper compatible de OpenAI.

In [0]:
%pip install -q langgraph databricks-sdk[openai]
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Lista de modelos disponibles:

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Lista todos los serving endpoints del workspace
for ep in w.serving_endpoints.list():
    print(ep.name)

databricks-claude-opus-4-8
databricks-gemini-3-5-flash
databricks-gpt-oss-120b
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-qwen35-122b-a10b
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b


## 2. Imports

In [0]:
from typing import TypedDict                            # Tipado del estado que viaja entre nodos del grafo.
from langgraph.graph import StateGraph, START, END      # Primitivas de LangGraph para construir el flujo
from databricks.sdk import WorkspaceClient              # Es el SDK oficial de Databricks para obtener un cliente OpenAI-compatible e invocar el modelo Fundacional
from concurrent.futures import ThreadPoolExecutor       # Ejecuta las llamadas al LLM en paralelo (cada comentario es independiente)

## 3. Cliente del LLM

Apuntamos al endpoint del Foundational Model `databricks-meta-llama-3-1-8b-instruct`.

In [0]:
w = WorkspaceClient()
client = w.serving_endpoints.get_open_ai_client()
ENDPOINT = "databricks-meta-llama-3-1-8b-instruct"

## 4. Definición del estado y del nodo de clasificación

En LangGraph cada nodo recibe un **estado** (un diccionario tipado) y devuelve una nueva versión del mismo. Aquí el estado lleva el comentario original y la etiqueta resultante (`0` o `1`).

El **prompt** define la tarea: el LLM debe responder únicamente con `1` (reclamo de seguridad) o `0` (cualquier otro tema). Forzar una salida mínima reduce el riesgo de respuestas verbosas y facilita el parseo.

El nodo invoca el endpoint vía `client.chat.completions.create()` (OpenAI-compatible format) con el formato chat estándar (lista de mensajes con `role` y `content`).

In [0]:
# Estructura del estado que viaja por el grafo
class State(TypedDict):
    comentario: str
    etiqueta: int

# Prompt de clasificación: instruye al LLM a responder solo con 0 o 1
PROMPT = """Eres un clasificador de reviews de billeteras móviles en Bolivia.

Tu tarea: determinar si el siguiente comentario negativo expresa un reclamo, queja o preocupación
relacionada con SEGURIDAD del usuario o su capital. Considera reclamos de seguridad:
- Suplantación de identidad / robo de cuenta
- Hackeo, accesos no autorizados, vulneración de credenciales
- Fraude, estafa, phishing, ingeniería social
- Pérdida o robo de dinero por terceros desconocidos
- Transacciones no reconocidas, cargos no autorizados
- Problemas con autenticación, OTP, biometría que comprometan seguridad

NO son reclamos de seguridad: lentitud, bugs de UI, problemas de conexión,
demoras en transferencias normales, atención al cliente, comisiones, etc.

Responde ÚNICAMENTE con un dígito, sin texto adicional:
1 = SÍ es reclamo de seguridad
0 = NO es reclamo de seguridad

Comentario: {comentario}
"""

# Nodo único del grafo: recibe el estado, invoca al LLM y devuelve la etiqueta
def clasificar(state: State) -> State:
    respuesta = client.chat.completions.create(
        model=ENDPOINT,
        messages=[{"role": "user", 
                   "content": PROMPT.format(comentario=state["comentario"])}],
        max_tokens=5,
        temperature=0,
    )
    texto = respuesta.choices[0].message.content
    etiqueta = 1 if "1" in texto.strip()[:3] else 0
    return {"comentario": state["comentario"], "etiqueta": etiqueta}

## 5. Construcción del grafo en LangGraph

Grafo de un solo nodo:

`START → clasificar → END`

In [0]:
graph = StateGraph(State)
graph.add_node("clasificar", clasificar)
graph.add_edge(START, "clasificar")
graph.add_edge("clasificar", END)

# Compilamos el grafo: lo deja listo para invocarse como una función
app = graph.compile()

## 6. Función wrapper para procesar un comentario

Encapsula la invocación del grafo. Recibe un string y devuelve `0` o `1`. Esta función es la que aplicaremos a cada fila del DataFrame.

In [0]:
import time

def procesar_comentario(texto: str) -> int:
    time.sleep(0.5)  # 500ms delay = max 2 requests/second per worker. Agrega delay para que no se exceda el limite de requests por segundo a la base de datos.
    resultado = app.invoke({"comentario": texto, "etiqueta": 0})
    return resultado["etiqueta"]

## 7. Carga de la tabla origen

Leemos la tabla Delta y la traemos a Pandas. Para 28k filas el `toPandas()` es razonable y permite usar `ThreadPoolExecutor` para paralelizar las llamadas HTTP al endpoint del LLM (cuello de botella de I/O, no de CPU).

In [0]:
df = spark.table("workspace.default.googleplay_apps_reviews").toPandas()

## 8. Clasificación en paralelo

Lanzamos múltiples hilos para invocar el endpoint del LLM concurrentemente. Como cada llamada es I/O-bound (espera la respuesta HTTP del modelo).

Esta llamada en paralelo no se refiere a los nodos Spark Workers, es acerca de los hilos Python que se desplegaran. El paralelismo es acerca las llamadas en la red, no sobre nodos de computacion distribuida.

Consejo:
* Limitar la cantidad de registros a analizar para evitar error de 'REQUEST_LIMIT_EXCEEDED'.
* `max_workers=10` Ir probando, 10 o menos lo recomendable por limitantes de licencia.

In [0]:
# Para no esperar a que el ThreadPoolExecutor falle con 28k en cola, se prueba primero.
print(procesar_comentario("me robaron la cuenta y perdí mi dinero"))
print(procesar_comentario("muy buena app"))

1
0


### Muestra
Se procesa una muestra por limitantes de ambiente y arquitectura.

In [0]:
# Se procesa una muestra del dataset.
df3 = df.head(10).copy()
df3["ReclamoSeguridad"] = df3["content"].apply(procesar_comentario)

# Se muestra
display(df3)

id,date,rating,version,content,Billetera,Store,ReclamoSeguridad
aeee0984-a69a-41a5-ab24-d025d0d7af67,2025-04-03,5.0,null,Muy bueno,Yolo,PlayStore,0
5ce00d12-cfcb-4d16-9c9d-a7192c05c360,2025-06-19,5.0,null,buena app,Yolo,PlayStore,0
2be101fc-1335-4721-a59e-605ffd37cdef,2024-07-10,5.0,null,Biem,Yolo,PlayStore,0
32ad758c-078c-407f-894e-25e29ce0ca51,2022-02-21,5.0,null,Excelente,Yolo,PlayStore,0
7c81b3d3-efaa-467c-8f71-591398f699cd,2022-07-28,5.0,null,Simple,Yolo,PlayStore,0
9538120c-d044-41ff-9fde-a7af8c453f61,2025-05-26,5.0,null,bien,Yolo,PlayStore,0
79c1a060-0553-4e37-8ff0-b7e05c470e7d,2026-04-22,5.0,null,esta buena,Yolo,PlayStore,0
83d47af2-f665-4a92-be78-66e8b2740d12,2025-12-07,5.0,null,Esta buena,Yolo,PlayStore,0
a408360a-a278-4cd6-844d-fcdca04ea1f3,2023-06-23,5.0,null,Funciona bien,Yolo,PlayStore,0
959e2f77-222b-4ac9-8254-a8c072779041,2024-12-06,5.0,null,Excelente,Yolo,PlayStore,0
